prepara la base y borra y recrea las dos colecciones:

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from database.setup_db import setup_database
from pipeline.limpiar import limpiar_csv
import pipeline.carga as pc

DB = "panteon"
db = setup_database(db_name=DB, drop_existing=True)

=== Inicializando Base de Datos: 'panteon' en mongodb://localhost:27017 ===
[*] Creando colección 'species' con validación $jsonSchema...
[*] Creando colección 'sequences' con validación $jsonSchema...

=== Configurando Índices Estratégicos ===
[*] Índices en colección 'species'...
[*] Índices en colección 'sequences'...

✓ Configuración completada con éxito.

--- Índices actuales en 'species' ---
 - _id_: SON([('_id', 1)])
 - idx_species_key_unique: SON([('species_key', 1)])
 - idx_kingdom_class: SON([('Kingdom', 1), ('Class', 1)])
 - idx_species_source: SON([('Source', 1)])

--- Índices actuales en 'sequences' ---
 - _id_: SON([('_id', 1)])
 - idx_sequence_id_unique: SON([('sequence_id', 1)])
 - idx_species_superfamily: SON([('species_key', 1), ('te_superfamily', 1)])
 - idx_te_hierarchy: SON([('te_class', 1), ('te_order', 1), ('te_superfamily', 1)])
 - idx_metrics_length: SON([('metrics.length', 1)])
 - idx_metrics_gc: SON([('metrics.gc_content', 1)])


carga las especies:

In [2]:
tabla = limpiar_csv("../data/PanTEon_Database_metadata_v1.6.1.csv")
insertadas, rechazadas = pc.cargar_species(db, tabla)
print("species:", insertadas, "insertadas |", len(rechazadas), "rechazadas")
print(rechazadas[:5])

species: 2790 insertadas | 0 rechazadas
[]


Celda 3, verifica:

In [3]:
sp = db.species
print("CSV limpio:", len(tabla), "| en MongoDB:", sp.count_documents({}))

for c in ["Phylum", "Class", "Order", "Family"]:
    print(c, "null:", sp.count_documents({c: {"$type": "null"}}))

for campo in ["Kingdom", "Source", "source_2"]:
    r = sp.aggregate([{"$group": {"_id": f"${campo}", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}])
    print(campo, {x["_id"]: x["n"] for x in r})

print("Índices:", list(sp.index_information()))

CSV limpio: 2790 | en MongoDB: 2790
Phylum null: 3
Class null: 49
Order null: 48
Family null: 76
Kingdom {'Fungi': 1559, 'Animalia': 780, 'Plantae': 451}
Source {'RM2+MCH': 2404, 'Dfam': 386}
source_2 {None: 2744, 'Repbase': 45, 'Dfam': 1}
Índices: ['_id_', 'idx_species_key_unique', 'idx_kingdom_class', 'idx_species_source']


In [4]:
from pipeline.fasta import leer_fasta

FASTA = "../data/PanTEon_Database_v1.6.1.fasta"

def secuencias_pendientes(especies=None):
    """Genera los documentos que aún no están en MongoDB."""
    ya = set(db.sequences.distinct("sequence_id"))
    for campos in leer_fasta(FASTA, especies):
        if campos["sequence_id"] not in ya:
            yield pc.armar_secuencia(campos)

pruebas

In [5]:
PRUEBA = {"armillaria cepistipes", "acomys cahirinus", "brachypodium distachyon"}
n, rech = pc.cargar_lotes(db.sequences, secuencias_pendientes(PRUEBA), "sequence_id")
print("prueba:", n, "insertadas |", len(rech), "rechazadas")

prueba: 290 insertadas | 0 rechazadas


In [11]:
   import importlib
   importlib.reload(pc)

<module 'pipeline.carga' from 'C:\\Users\\sebas\\OneDrive\\Escritorio\\UNIVERSIDAD\\BASES DE DATOS 2\\Corte2Proyecto\\pipeline\\carga.py'>

carga completa, cuidado con este que se quedan sin compu muchachos

In [ ]:
n, rech = pc.cargar_lotes(db.sequences, secuencias_pendientes(), "sequence_id", tam=1000, cada=10)
print("sequences:", n, "insertadas |", len(rech), "rechazadas")

if rech:
    import pandas as pd
    Path("../reports").mkdir(exist_ok=True)
    pd.DataFrame(rech, columns=["sequence_id", "codigo"]).to_csv("../reports/rechazos_sequences.csv", index=False)

print("Total en MongoDB:", db.sequences.count_documents({}))

NameError: name 'pc' is not defined

: 

conteos y huérfanas

In [13]:
import pandas as pd
from pipeline.limpiar import clave

fa = pd.read_csv("../data/fasta_resumen.csv", keep_default_na=False)
fa["species_key"] = fa["especie"].map(clave)

print("sequences:", db.sequences.count_documents({}), "| FASTA:", len(fa))
print("species:  ", db.species.count_documents({}))

en_seq = set(db.sequences.distinct("species_key"))
en_sp = set(db.species.distinct("species_key"))
print("species_key en sequences sin especie (huérfanas):", len(en_seq - en_sp))
print("Especies sin ninguna secuencia:", len(en_sp - en_seq))

sequences: 239946 | FASTA: 239946
species:   2790
species_key en sequences sin especie (huérfanas): 0
Especies sin ninguna secuencia: 0


agregaciones contra el resumen

In [14]:
mongo_sp = {x["_id"]: x["n"] for x in db.sequences.aggregate(
    [{"$group": {"_id": "$species_key", "n": {"$sum": 1}}}])}
fa_sp = fa.groupby("species_key").size().to_dict()
print("Especies con conteo distinto:", sum(mongo_sp.get(k) != v for k, v in fa_sp.items()))

mongo_cl = {x["_id"]: x["n"] for x in db.sequences.aggregate(
    [{"$group": {"_id": "$classification_raw", "n": {"$sum": 1}}}])}
fa_cl = fa["clasificacion"].value_counts().to_dict()
print("Clasificaciones con conteo distinto:", sum(mongo_cl.get(k) != v for k, v in fa_cl.items()),
      "| clasificaciones distintas:", len(fa_cl))

tot = next(db.sequences.aggregate([{"$group": {
    "_id": None, "largo": {"$sum": "$metrics.length"}, "n_N": {"$sum": "$metrics.n_count"}}}]))
print("Suma de longitudes:", tot["largo"], "| FASTA:", int(fa["longitud"].sum()))
print("Suma de N:         ", tot["n_N"], "| FASTA:", int(fa["n_N"].sum()))

print("gc_content que no es double:",
      db.sequences.count_documents({"metrics.gc_content": {"$not": {"$type": "double"}}}))

Especies con conteo distinto: 0
Clasificaciones con conteo distinto: 0 | clasificaciones distintas: 66
Suma de longitudes: 963901415 | FASTA: 963901415
Suma de N:          2147161 | FASTA: 2147161
gc_content que no es double: 0


relectura de una muestra contra el FASTA original. Toma 200 secuencias al azar y además las de más y menos longitud, la de mayor porcentaje de N (el 50 %) y 5 con ID sin prefijo PDB:

In [2]:
import random
from collections import Counter
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

fa["pct_N"] = fa["n_N"] / fa["longitud"] * 100
azar = random.Random(7).sample(list(fa["id"]), 200)
extremos = [fa.loc[fa["longitud"].idxmax(), "id"], fa.loc[fa["longitud"].idxmin(), "id"],
            fa.loc[fa["pct_N"].idxmax(), "id"]]
no_pdb = list(fa.loc[~fa["id"].str.startswith("PDB"), "id"].head(5))
ids = set(azar + extremos + no_pdb)

originales = {}
for rec in SeqIO.parse(FASTA, "fasta"):
    id_ = rec.description.split("#", 1)[0].strip()
    if id_ in ids:
        originales[id_] = str(rec.seq).upper()
        if len(originales) == len(ids):
            break

def ref_kmers(s, k):
    return dict(Counter(s[i:i+k] for i in range(len(s) - k + 1) if "N" not in s[i:i+k]))

difs, tipos_malos = [], 0
for sid, s in originales.items():
    g = db.sequences.find_one({"sequence_id": sid})
    if g["sequence"] != s: difs.append((sid, "sequence"))
    m = g["metrics"]
    if m["length"] != len(s): difs.append((sid, "length"))
    if m["n_count"] != s.count("N"): difs.append((sid, "n_count"))
    if abs(m["gc_content"] - gc_fraction(s) * 100) > 1e-9: difs.append((sid, "gc"))
    for k in range(1, 7):
        if g["kmers"][f"k{k}"] != ref_kmers(s, k): difs.append((sid, f"k{k}"))
        tipos_malos += sum(type(v) is not int for v in g["kmers"][f"k{k}"].values())
    tipos_malos += type(m["gc_content"]) is not float
print("revisadas:", len(originales), "de", len(ids), "| diferencias:", difs, "| tipos raros:", tipos_malos)

NameError: name 'fa' is not defined

: tamaño en disco

In [16]:
for nombre in ("species", "sequences"):
    st = db.command("collStats", nombre)
    print(f"{nombre}: {st['count']:,} docs | medio {st['avgObjSize']/1024:.1f} KB | "
          f"sin comprimir {st['size']/1024**3:.2f} GB | en disco {st['storageSize']/1024**3:.2f} GB | "
          f"índices {st['totalIndexSize']/1024**2:.1f} MB")

species: 2,790 docs | medio 0.3 KB | sin comprimir 0.00 GB | en disco 0.00 GB | índices 0.2 MB
sequences: 239,946 docs | medio 36.7 KB | sin comprimir 8.40 GB | en disco 3.42 GB | índices 22.3 MB


cruce fasta y csv

In [4]:
solo_fasta = set(fa["especie"].map(clave)) - set(tabla["species_key"])
solo_csv = set(tabla["species_key"]) - set(fa["especie"].map(clave))
print("Especies del FASTA sin metadatos en el CSV:", len(solo_fasta))
print("Especies del CSV sin secuencias en el FASTA:", len(solo_csv))

NameError: name 'fa' is not defined